# 🔧 02 — Pipeline Development

**Goal:** Build and iterate on the candidate ranking pipeline step-by-step.

**Prerequisites:**
- Completed `01_eda.ipynb` (you understand the data)
- Updated `configs/config.yaml` with actual column names

**Workflow:**
1. Load & preprocess data
2. Build text representations
3. Generate embeddings
4. Compute semantic similarity
5. Engineer structured features
6. Combine into hybrid scores
7. Evaluate ranking quality

---

## 0. Setup

In [ ]:
# --- Colab Setup (uncomment if running in Colab) ---
# !pip install -q -r requirements.txt
# from src.utils import setup_colab
# setup_colab()

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_config, load_candidates, load_jobs
from src.preprocess import build_candidate_text, build_job_text, clean_text
from src.embeddings import load_embedding_model, generate_embeddings, compute_similarity
from src.features import build_feature_vector
from src.scoring import compute_hybrid_score, score_all_candidates, explain_score
from src.ranker import rank_candidates, format_shortlist
from src.utils import timer, print_ranking

print('✅ All modules imported')

In [ ]:
# Load config
config = load_config('../configs/config.yaml')

# TODO: Load your actual data
# candidates = load_candidates(config)
# jobs = load_jobs(config)

---

## Step 1: Text Preprocessing

Build composite text representations for candidates and jobs.

**Key idea:** Combine multiple fields (title, skills, resume, education) into one
text string that the embedding model can encode holistically.

In [ ]:
# TODO: Build text representations

# candidate_texts = [
#     build_candidate_text(candidates.iloc[i], config)
#     for i in range(len(candidates))
# ]
# 
# # Pick a job to rank for
# job_row = jobs.iloc[0]
# job_text = build_job_text(job_row, config)
# 
# # Inspect
# print('Sample candidate text:')
# print(candidate_texts[0][:300])
# print('\nJob text:')
# print(job_text[:300])

---

## Step 2: Embedding Generation

Use sentence-transformers to encode texts into dense vectors.

**Note:** First run will download the model (~80MB for MiniLM).

In [ ]:
# Load embedding model
# model = load_embedding_model(config['embeddings']['model_name'])

In [ ]:
# Generate embeddings
# candidate_embeddings = generate_embeddings(candidate_texts, model, batch_size=32)
# job_embedding = generate_embeddings([job_text], model)
# 
# print(f'Candidate embeddings shape: {candidate_embeddings.shape}')
# print(f'Job embedding shape: {job_embedding.shape}')
# 
# # Optional: Save embeddings to avoid re-computing
# from src.utils import save_embeddings
# save_embeddings(candidate_embeddings, '../outputs/models/candidate_embeddings.npy')

---

## Step 3: Semantic Similarity (Baseline Ranking)

Compute cosine similarity between job embedding and each candidate.

This gives us a **baseline semantic ranking** — candidates ordered by
how semantically similar their profile is to the JD.

In [ ]:
# Compute semantic similarity
# semantic_scores = compute_similarity(job_embedding, candidate_embeddings)
# 
# print(f'Scores shape: {semantic_scores.shape}')
# print(f'Mean: {semantic_scores.mean():.4f}')
# print(f'Min:  {semantic_scores.min():.4f}')
# print(f'Max:  {semantic_scores.max():.4f}')
# 
# # Visualize distribution
# plt.figure(figsize=(10, 4))
# plt.hist(semantic_scores, bins=50, color='steelblue', edgecolor='white')
# plt.title('Semantic Similarity Score Distribution')
# plt.xlabel('Cosine Similarity')
# plt.ylabel('Count')
# plt.show()

---

## Step 4: Structured Feature Engineering

Extract features a recruiter would care about:
- Skills overlap with JD requirements
- Experience match
- Education alignment

**TODO:** Add more features as you understand the data better.

In [ ]:
# Build feature vectors for all candidates
# all_features = []
# for i in range(len(candidates)):
#     feat = build_feature_vector(candidates.iloc[i], job_row, config)
#     all_features.append(feat)
# 
# # Convert to DataFrame for easy inspection
# features_df = pd.DataFrame(all_features)
# print(features_df.describe())
# features_df.head(10)

---

## Step 5: Hybrid Scoring

Combine semantic similarity + structured features into one final score.

**Weights** are defined in `configs/config.yaml` → `scoring` section.
Tune these based on what seems to work best qualitatively.

In [ ]:
# Compute hybrid scores
# weights = config['scoring']
# hybrid_scores = score_all_candidates(semantic_scores, all_features, weights)
# 
# print(f'Hybrid scores — min: {min(hybrid_scores):.4f}, max: {max(hybrid_scores):.4f}')
# 
# # Compare semantic vs hybrid
# plt.figure(figsize=(10, 5))
# plt.scatter(semantic_scores, hybrid_scores, alpha=0.5, s=10)
# plt.xlabel('Semantic Score')
# plt.ylabel('Hybrid Score')
# plt.title('Semantic vs Hybrid Score Correlation')
# plt.show()

---

## Step 6: Ranking & Evaluation

Produce the final ranked shortlist and inspect the results.

In [ ]:
# Rank candidates
# cc = config['dataset']['candidate_columns']
# ranked = rank_candidates(candidates, hybrid_scores, top_k=10, id_column=cc['id'])
# shortlist = format_shortlist(ranked, config, include_explanation=True)
# 
# print_ranking(ranked)
# shortlist

In [ ]:
# Explain individual candidate scores
# top_idx = 0  # Look at the top-ranked candidate
# breakdown = explain_score(semantic_scores[top_idx], all_features[top_idx], weights)
# 
# print('Score breakdown for top candidate:')
# for component, value in breakdown.items():
#     print(f'  {component}: {value:.4f}')

---

## Step 7: Iterate & Improve

**TODO:**
- [ ] Tune scoring weights in `config.yaml`
- [ ] Add more structured features in `src/features.py`
- [ ] Try different embedding models
- [ ] Qualitatively evaluate: do the top candidates make sense?
- [ ] If labels available: train supervised re-ranker

### Experiments to try:
| Experiment | Config Change | Hypothesis |
|-----------|---------------|------------|
| Higher semantic weight | `semantic_weight: 0.7` | Better for broad role matching |
| Higher skills weight | `skills_weight: 0.4` | Better for technical roles |
| Better model | `model_name: all-mpnet-base-v2` | Higher quality embeddings |
| ... | ... | ... |